# Figure3c activity volcano


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patheffects as pe
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from adjustText import adjust_text
from statsmodels.stats.multitest import multipletests
import os
import warnings
warnings.filterwarnings("ignore")

INPUT_CSV = "results/permutation_full/high_vs_low_mean/highlow_mean_gene_level_clean_summary.csv"
OUTPUT_STEM = "results/permutation_volcano_publication/high_vs_low/highlow_mean_volcano_fisher_zscore"

EFFECT_COL = "z_score"
PVAL_COL   = "fisher_perm_p"
QVAL_COL   = "fisher_qval"
PROP_COL   = "prop_high_case_donors"

Q_THRESHOLD     = 0.1
PROP_CAP        = 0.10
LABEL_TOP_N_POS = 20

CMAP_RED       = "Reds"
FIGSIZE        = (11, 7)
FIGURE_DPI     = 300
LABEL_FONTSIZE = 9
AXIS_FONTSIZE  = 11
TITLE_FONTSIZE = 12

DOT_SIZE_GENERIC  = 22
DOT_SIZE_CAT      = 38
DOT_SIZE_NS       = 8

plt.rcParams.update({
    "font.family":     "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":       10,
    "axes.linewidth":  1.0,
    "axes.labelsize":  AXIS_FONTSIZE,
    "axes.titlesize":  TITLE_FONTSIZE,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8.5,
    "savefig.dpi":     FIGURE_DPI,
    "savefig.bbox":    "tight",
})

CATEGORIES = [
    ("Nuclear Pore",
     {"NPIPB11", "NPIPB15", "NPIPB4", "NPIPB3", "NUPL1", "NUP58"},
     "#5C2D91"),
    ("Histone Related",
     {"H3F3C", "H3F3A", "HIST1H4G", "KMT2C", "INO80E", "MEAF6",
      "BAHCC1", "TOP3A", "UIMC1", "SMYD5"},
     "#C71585"),
    ("Spliceosome",
     {"SNRNP70", "SNRPD2", "PRPF38B", "SFSWAP", "RBM25", "SCAF8",
      "SON", "SRSF5", "SRRT", "SAFB", "ACIN1", "NELFE", "ELL",
      "ZC3H13", "MEX3D", "UPF3B", "PDCD11"},
     "#7B0000"),
    ("Cell Death",
     {"ACIN1", "DIDO1", "XKR7", "PDCD11", "HUWE1", "CCAR1",
      "CACTIN", "UIMC1"},
     "#B8860B"),
]

gene_to_cat = {}
for cat_name, gene_set, cat_color in CATEGORIES:
    for g in gene_set:
        if g not in gene_to_cat:
            gene_to_cat[g] = (cat_name, cat_color)

cell_death_genes = {g for cat_name, gene_set, _ in CATEGORIES
                    if cat_name == "Cell Death" for g in gene_set}

print("[1] Loading data...")
df = pd.read_csv(INPUT_CSV)
df = df.dropna(subset=[EFFECT_COL, PVAL_COL]).copy().reset_index(drop=True)
print(f"    {len(df):,} genes")

pvals = np.clip(df[PVAL_COL].values, 1e-10, 1.0)
df["neg_log10_p"] = -np.log10(pvals)

if QVAL_COL in df.columns and df[QVAL_COL].notna().any():
    df["is_sig"] = df[QVAL_COL] <= Q_THRESHOLD
else:
    _, qvals, _, _ = multipletests(pvals, method="fdr_bh")
    df[QVAL_COL] = qvals
    df["is_sig"] = qvals <= Q_THRESHOLD

n_sig = df["is_sig"].sum()
print(f"    Significant (FDR < {Q_THRESHOLD}): {n_sig}")

sig_df    = df[df[QVAL_COL] <= Q_THRESHOLD].copy()
genes_pos = (sig_df[sig_df[EFFECT_COL] > 0]
             .sort_values(EFFECT_COL, ascending=False)
             .head(LABEL_TOP_N_POS)["gene"].tolist())
genes_neg = (sig_df[sig_df[EFFECT_COL] < 0]
             .sort_values(EFFECT_COL, ascending=True)["gene"].tolist())

sig_cell_death = (sig_df[sig_df["gene"].isin(cell_death_genes) & (sig_df[EFFECT_COL] > 0)]
                  .sort_values(EFFECT_COL, ascending=False)["gene"].tolist())

genes_to_label = list(dict.fromkeys(genes_pos + sig_cell_death + genes_neg))
n_cd_extra = len([g for g in sig_cell_death if g not in genes_pos])
print(f"    Labels: {len(genes_to_label)} "
      f"({len(genes_pos)} top-N HIGH + {n_cd_extra} extra Cell Death + {len(genes_neg)} LOW)")

cmap_r   = cm.get_cmap(CMAP_RED)
has_prop = PROP_COL in df.columns and df[PROP_COL].notna().any()

effect_vals = df[EFFECT_COL].values
neg_log10_p = df["neg_log10_p"].values
is_sig_arr  = df["is_sig"].values

colors      = []
sizes       = []
point_types = []

for i in range(len(df)):
    gene = df.iloc[i]["gene"]
    if not is_sig_arr[i]:
        colors.append("lightgray")
        sizes.append(DOT_SIZE_NS)
        point_types.append("ns")
    elif gene in gene_to_cat:
        _, cat_color = gene_to_cat[gene]
        colors.append(cat_color)
        sizes.append(DOT_SIZE_CAT)
        point_types.append("sig_cat")
    elif effect_vals[i] > 0:
        point_types.append("sig_pos")
        sizes.append(DOT_SIZE_GENERIC)
        if has_prop:
            prop = df.iloc[i][PROP_COL]
            prop = 0.0 if pd.isna(prop) else prop
            prop_scaled = np.clip(prop, 0.0, PROP_CAP) / PROP_CAP
            colors.append(cmap_r(0.15 + 0.85 * prop_scaled))
        else:
            colors.append("firebrick")
    else:
        colors.append("steelblue")
        sizes.append(DOT_SIZE_GENERIC)
        point_types.append("sig_neg")

point_types = np.array(point_types)
sizes       = np.array(sizes)
n_pos = ((point_types == "sig_pos") | (point_types == "sig_cat")).sum()
n_neg = (point_types == "sig_neg").sum()

print("[2] Drawing...")
fig, ax = plt.subplots(figsize=FIGSIZE)

ns_idx = np.where(point_types == "ns")[0]
ax.scatter(effect_vals[ns_idx], neg_log10_p[ns_idx],
           c="lightgray", s=DOT_SIZE_NS, alpha=0.35, edgecolors="none",
           zorder=1, rasterized=True)

neg_idx = np.where(point_types == "sig_neg")[0]
if len(neg_idx):
    ax.scatter(effect_vals[neg_idx], neg_log10_p[neg_idx],
               c="steelblue", s=DOT_SIZE_GENERIC, marker="o",
               alpha=0.85, edgecolors="white", linewidths=0.5, zorder=2)

pos_idx = np.where(point_types == "sig_pos")[0]
if len(pos_idx):
    c_vals = [colors[i] for i in pos_idx]
    ax.scatter(effect_vals[pos_idx], neg_log10_p[pos_idx],
               c=c_vals, s=DOT_SIZE_GENERIC, marker="o",
               alpha=0.9, edgecolors="white", linewidths=0.5, zorder=3)

cat_idx = np.where(point_types == "sig_cat")[0]
if len(cat_idx):
    c_vals = [colors[i] for i in cat_idx]
    ax.scatter(effect_vals[cat_idx], neg_log10_p[cat_idx],
               c=c_vals, s=DOT_SIZE_CAT, marker="o",
               alpha=0.92, edgecolors="white", linewidths=0.6, zorder=4)

ax.axvline(x=0, color="black", lw=0.8, alpha=0.5)
for p_thresh, ls in [(0.05, ":"), (0.01, "--"), (0.001, "-")]:
    ax.axhline(-np.log10(p_thresh), color="gray", ls=ls, lw=0.6, alpha=0.35)

ax.set_xlabel("Z-score (log(OR) / SE)", fontsize=AXIS_FONTSIZE, fontweight="medium")
ax.set_ylabel(r"$-\log_{10}$(Fisher permutation p-value)",
              fontsize=AXIS_FONTSIZE, fontweight="medium")
ax.set_title("HIGH vs LOW SLEDAI (MEAN Aggregation)\nFisher Combined",
             fontsize=TITLE_FONTSIZE, fontweight="bold", pad=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, ls="--", alpha=0.12, zorder=0)

if has_prop and n_pos > 0:
    sm = cm.ScalarMappable(cmap=cmap_r, norm=Normalize(vmin=0, vmax=PROP_CAP))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, location="right", shrink=0.45, pad=0.02, aspect=20)
    cbar.set_label("Prop. HIGH SLEDAI donors with z ≥ 4.5", fontsize=8.5)
    ticks = np.linspace(0, PROP_CAP, 5)
    tick_labels = [f"{v*100:.0f}%" for v in ticks]
    tick_labels[-1] = f"≥{PROP_CAP*100:.0f}%"
    cbar.set_ticks(ticks); cbar.set_ticklabels(tick_labels)

red_fill = cmap_r(0.75)
legend_elements = [
    Line2D([0],[0], marker="o", color="w", markerfacecolor=red_fill,
           markersize=8, markeredgecolor="white", markeredgewidth=0.4,
           label=f"HIGH-enriched (FDR < {Q_THRESHOLD})"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="steelblue",
           markersize=8, markeredgecolor="white", markeredgewidth=0.4,
           label=f"LOW-enriched (FDR < {Q_THRESHOLD})"),
    Line2D([0],[0], marker="o", color="w", markerfacecolor="lightgray",
           markersize=7, label="Not Significant"),
]
for cat_name, _, cat_color in CATEGORIES:
    legend_elements.append(
        Line2D([0],[0], marker="o", color="w",
               markerfacecolor=cat_color, markeredgecolor="white",
               markeredgewidth=0.4, markersize=10, label=cat_name)
    )
ax.legend(handles=legend_elements, loc="lower left", frameon=True,
          framealpha=0.92, edgecolor="gray", fontsize=8.5,
          handletextpad=0.5, borderpad=0.6)

summary_text = (
    f"n = {len(df):,} genes\n"
    f"Significant (FDR < {Q_THRESHOLD}): {n_sig}\n"
    f"  HIGH-enriched: {n_pos}\n"
    f"  LOW-enriched:  {n_neg}"
)
ax.text(0.985, 0.02, summary_text, transform=ax.transAxes,
        ha="right", va="bottom", fontsize=7.5, family="monospace",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="wheat",
                  edgecolor="gray", alpha=0.85))

def get_txt_color(gene, pt):
    if gene in gene_to_cat:
        return gene_to_cat[gene][1]
    return cmap_r(0.92) if pt == "sig_pos" else "steelblue"

text_data_coords = []
texts = []
for gene in genes_to_label:
    mask = df["gene"] == gene
    if not mask.any(): continue
    row = df.loc[mask].iloc[0]
    idx  = mask.values.argmax()
    pt   = point_types[idx]
    txt_color = get_txt_color(gene, pt)
    t = ax.text(row[EFFECT_COL], row["neg_log10_p"], gene,
                fontsize=LABEL_FONTSIZE, ha="left", va="bottom",
                fontweight="bold", color=txt_color, zorder=15)
    t.set_path_effects([pe.withStroke(linewidth=2.2, foreground="white")])
    texts.append(t)
    text_data_coords.append((t, row[EFFECT_COL], row["neg_log10_p"]))

if texts:
    try:
        adjust_text(
            texts, ax=ax,
            expand_points=(1.6, 1.9), expand_text=(1.3, 1.5),
            force_points=(0.5, 0.9), force_text=(0.4, 0.6),
            only_move={"points": "y", "texts": "xy"},
            lim=600,
        )
    except Exception as e:
        print(f"  [WARN] adjust_text: {e}")

fig.canvas.draw()
for t, x_pt, y_pt in text_data_coords:
    x_txt, y_txt = t.get_position()
    dist = np.sqrt((x_txt - x_pt)**2 + (y_txt - y_pt)**2)
    if dist > 0.08:
        ax.plot([x_pt, x_txt], [y_pt, y_txt],
                color="dimgray", lw=0.9, alpha=0.65, zorder=5,
                solid_capstyle="round")

plt.tight_layout()
os.makedirs(os.path.dirname(OUTPUT_STEM), exist_ok=True)
png_path = OUTPUT_STEM + ".png"
pdf_path = OUTPUT_STEM + ".pdf"
fig.savefig(png_path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, format="pdf", bbox_inches="tight", facecolor="white")
plt.close()

print(f"\n[DONE]")
print(f"  PNG: {png_path}")
print(f"  PDF: {pdf_path}")